# Shiksha Setu Attendance Analysis
Veltris Challenge A · Shiva Sai Vemula

Source: assessment datasets A1–A3. Run from the submission directory or from the analysis directory.

In [1]:
"""Reproduce Challenge A tables from the assessment CSV files. Python 3.10+."""
import csv
from pathlib import Path
from statistics import correlation
import sys

folder = Path.cwd()
if not (folder / "schools.csv").is_file():
    folder = folder / "analysis"
sys.path.insert(0, str(folder.parent / "prototype"))
from risk_engine import rank_students


def read_table(name):
    with (folder / name).open(newline="", encoding="utf-8") as source:
        return [{k: (v if k == "id" else int(v)) for k, v in row.items()}
                for row in csv.DictReader(source)]




## A1 School attendance
Estimate daily absences as enrollment multiplied by one minus the attendance rate. Divide total estimates by district enrollment to obtain the district absence rate.

In [2]:
schools = read_table("schools.csv")
students = read_table("students.csv")
enrolled = sum(s["enrolled"] for s in schools)
absences = [s["enrolled"] * (100 - s["attendance"]) / 100 for s in schools]
lowest = sorted(range(len(schools)), key=lambda i: schools[i]["attendance"])[:3]
print(f"Enrollment: {enrolled}; weighted attendance: {100*(1-sum(absences)/enrolled):.2f}%")
print(f"Estimated daily absences: {sum(absences):.1f}")
print(f"Lowest three schools: {sum(absences[i] for i in lowest)/sum(absences):.1%} of estimated absences")


Enrollment: 900; weighted attendance: 71.03%
Estimated daily absences: 260.7
Lowest three schools: 39.3% of estimated absences


## A2 Associations
The twelve records are synthetic. These associations describe the sample. The effect of distance and prior visits requires longitudinal records and comparisons over the same weeks.

In [3]:
for field in ("score", "km", "visits"):
    r = correlation([s["attendance"] for s in students], [s[field] for s in students])
    print(f"Attendance vs {field}, Pearson r: {r:.3f}")
print(f"Score vs distance, Pearson r: {correlation([s['score'] for s in students], [s['km'] for s in students]):.3f}")


Attendance vs score, Pearson r: 0.950
Attendance vs km, Pearson r: -0.958
Attendance vs visits, Pearson r: -0.668
Score vs distance, Pearson r: -0.932


## A2 Priority and recommended actions
Reproduce the report ranking with the risk engine included in the prototype.

In [4]:
records = [dict(student_id=s["id"], school_id=s["school"], attendance=s["attendance"],
                prev_score=s["score"], distance_km=s["km"], prior_visits=s["visits"]) for s in students]
baselines = [dict(school_id=s["school"], avg_attendance=s["attendance"]) for s in schools]
ranked = rank_students(records, baselines)
print("Student   Points   Action")
for row in ranked:
    print(f"{row.student_id:8} {row.total_score:6.2f}   {row.action_channel}")


Student   Points   Action
ST009     88.91   case_review
ST004     75.67   case_review
ST012     73.55   home_visit
ST007     67.21   home_visit
ST006     51.55   teacher_call
ST002     47.58   teacher_call
ST011     46.24   teacher_call
ST003     23.63   monitor
ST010     22.03   monitor
ST005      6.64   monitor
ST008      2.50   monitor
ST001      2.50   monitor


## A3 Staffing and budget
The visit limit is the smaller of the staffing ceiling and the funded ceiling. A 10% planning reserve leaves ten slots and INR 2,500 for urgent needs.

In [5]:
print(f"Physical ceiling: {5*4*5} visits; funded ceiling: {25000//250} visits")
print(f"With 10% reserve: 90 planned visits, INR {90*250:,} planned spend")


Physical ceiling: 100 visits; funded ceiling: 100 visits
With 10% reserve: 90 planned visits, INR 22,500 planned spend
